<a href="https://colab.research.google.com/github/antaleksandrgit-oss/spaceship-titanic/blob/main/notebooks/spaceship_titanic_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib as plt


test = pd.read_csv('/content/test.csv')
train = pd.read_csv('/content/train.csv')

df = pd.concat([test, train])
df["Transported"] = df["Transported"].map({True: 1, False: 0})
transported = df[["PassengerId", "Transported"]].copy()
rem_df = df.drop(['PassengerId', 'Transported', 'Name', 'Cabin', "HomePlanet", 'Destination', "CryoSleep"], axis=1)


rem_df["RoomService"] = np.log1p(rem_df["RoomService"])
rem_df["ShoppingMall"] = np.log1p(rem_df["ShoppingMall"])
rem_df["Spa"] = np.log1p(rem_df["Spa"])
rem_df["VRDeck"] = np.log1p(rem_df["VRDeck"])
rem_df["Age"] = np.log1p(rem_df["Age"])
rem_df["FoodCourt"] = np.log1p(rem_df["FoodCourt"])

rem_df["VIP"] = rem_df["VIP"].map({True: 1, False: 0})
#rem_df["CryoSleep"] = rem_df["CryoSleep"].map({True: 1, False: 0})

#rem_df["CryoSleep"].infer_objects(copy=False)

#rem_df["CryoSleep"] = rem_df["CryoSleep"].fillna(0)
rem_df["VIP"] = rem_df["VIP"].fillna(0)
rem_df["RoomService"] = rem_df["RoomService"].fillna(0)
rem_df["Spa"] = rem_df["Spa"].fillna(0)
rem_df["FoodCourt"] = rem_df["FoodCourt"].fillna(0)
rem_df["ShoppingMall"] = rem_df["ShoppingMall"].fillna(0)
rem_df["VRDeck"] = rem_df["VRDeck"].fillna(0)
rem_df["Age"] = rem_df["Age"].fillna(rem_df["Age"].median())

#rem_df_encoding = pd.get_dummies(rem_df, columns=['HomePlanet', 'Destination'])
rem_df_encoding = rem_df.copy()

train_x, train_y = rem_df_encoding[4277:], transported["Transported"][4277:]
test_x, test_y = rem_df_encoding[:4277], transported["Transported"][:4277]

print(rem_df_encoding)

           Age  VIP  RoomService  FoodCourt  ShoppingMall       Spa    VRDeck
0     3.332205  0.0     0.000000   0.000000      0.000000  0.000000  0.000000
1     2.995732  0.0     0.000000   2.302585      0.000000  7.945910  0.000000
2     3.465736  0.0     0.000000   0.000000      0.000000  0.000000  0.000000
3     3.663562  0.0     0.000000   8.802823      0.000000  5.204007  6.373320
4     3.044522  0.0     2.397895   0.000000      6.455199  0.000000  0.000000
...        ...  ...          ...        ...           ...       ...       ...
8688  3.737670  1.0     0.000000   8.827615      0.000000  7.404888  4.317488
8689  2.944439  0.0     0.000000   0.000000      0.000000  0.000000  0.000000
8690  3.295837  0.0     0.000000   0.000000      7.535297  0.693147  0.000000
8691  3.496508  0.0     0.000000   6.956545      0.000000  5.869297  8.082093
8692  3.806662  0.0     4.844187   8.452975      0.000000  0.000000  2.564949

[12970 rows x 7 columns]


In [ ]:
correletion_martix = rem_df_encoding.corr()

print(correletion_martix)

                   Age       VIP  RoomService  FoodCourt  ShoppingMall  \
Age           1.000000  0.074260     0.158558   0.210834      0.144481   
VIP           0.074260  1.000000     0.056655   0.117829      0.046577   
RoomService   0.158558  0.056655     1.000000   0.088583      0.361728   
FoodCourt     0.210834  0.117829     0.088583   1.000000      0.099745   
ShoppingMall  0.144481  0.046577     0.361728   0.099745      1.000000   
Spa           0.206377  0.090673     0.148051   0.414398      0.163594   
VRDeck        0.194646  0.095989     0.097802   0.455259      0.101445   

                   Spa    VRDeck  
Age           0.206377  0.194646  
VIP           0.090673  0.095989  
RoomService   0.148051  0.097802  
FoodCourt     0.414398  0.455259  
ShoppingMall  0.163594  0.101445  
Spa           1.000000  0.379493  
VRDeck        0.379493  1.000000  


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier

model1 = RandomForestClassifier(n_estimators=100, random_state=42)
model2 = XGBClassifier(n_estimators=200, learning_rate=0.05)
model3 = LGBMClassifier(n_estimators=200, learning_rate=0.05)

ensemble = VotingClassifier([
    ('rf', model1),
    ('xgb', model2),
    ('lgb', model3)
], voting='soft')

ensemble.fit(train_x, train_y)

predict = ensemble.predict(test_x)

predict = predict.astype(bool)

submission = pd.DataFrame({
    'PassengerId': transported["PassengerId"][:4277],
    'Transported': predict
})
print(submission)
submission.to_csv('submission.csv', index=False)
print("\nФайл 'my_first_submission.csv'")

[LightGBM] [Info] Number of positive: 4378, number of negative: 4315
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001484 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1356
[LightGBM] [Info] Number of data points in the train set: 8693, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.503624 -> initscore=0.014495
[LightGBM] [Info] Start training from score 0.014495
     PassengerId  Transported
0        0013_01         True
1        0018_01        False
2        0019_01         True
3        0021_01         True
4        0023_01        False
...          ...          ...
4272     9266_02         True
4273     9269_01        False
4274     9271_01         True
4275     9273_01         True
4276     9277_01         True

[4277 rows x 2 columns]

Файл 'my_first_submission.csv'
